# Неделя 4. Transfer Learning — семинар с пошаговыми объяснениями

Этот ноутбук — **полный учебный пример без оценки**. Мы адаптируем предобученный энкодер BERT для распознавания именованных сущностей (**NER**). В домашнем задании те же инструменты используются для другой задачи: поиска дубликатов вопросов. Код этого семинара можно повторно использовать в домашней работе. Сдавать нужно **только домашнее задание** с отчётом об экспериментах.

Как выглядит метод переноса обучения из лекции:

- **Исходная задача:** masked language modeling на неразмеченных текстах. Этот этап уже выполнен авторами BERT; мы загружаем их веса.
- **Целевая задача:** NER с разметкой примерно 15% обучающих предложений CoNLL-2003.
- **Адаптация:** новая выходная «голова» поверх предобученного энкодера, затем дообучение всей сети.

| Раздел | Соответствующая часть лекции |
| --- | --- |
| 1. Токены, padding и маски внимания | Входные представления BERT |
| 2. Предобученный энкодер и выходная голова | BERT как Transformer-энкодер и masked LM |
| 3. Данные для NER | Задачи на уровне токенов |
| 4–5. Обучаемые параметры и fine-tuning | Новая голова и способы использования предобученной сети |
| 6. Вклад предобучения | Transfer learning |
| 7. Связь с T5 | Переиспользование предобученного выходного слоя |

После выполнения ты должен суметь объяснить, откуда берутся предобученные веса, как разметка слов переносится на подслова и какие параметры получают градиенты. Каждый раздел начинается с основной идеи, а ячейки **«На что обратить внимание»** поясняют результаты.

> Неофициальный русский перевод [оригинала](./seminar.ipynb). Код сохранён.

In [ ]:
# В Colab уже установлены PyTorch, Transformers 5 и Datasets.
# В других средах установи PyTorch под CPU/GPU, затем transformers>=5.18,<6 и datasets>=3,<5.
import gc
import time
from collections import Counter

import datasets
import torch
import transformers
from torch.utils.data import DataLoader
from transformers import (
    AutoConfig,
    AutoModelForMaskedLM,
    AutoModelForTokenClassification,
    AutoTokenizer,
    DataCollatorForTokenClassification,
    pipeline,
    set_seed,
)

SEED = 42
BASE_MODEL = "google-bert/bert-base-cased"
MODEL_REVISION = "cd5ef92a9fb2f889e972770a36d4ed042daf221e"
DATASET_REVISION = "19edcb426bfd625c275c17b9a99b2239243f4377"
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
set_seed(SEED)
print("Versions:", torch.__version__, transformers.__version__, datasets.__version__)
print("Device:", DEVICE)
if DEVICE.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(DEVICE))

## 1. Токены, заполнение последовательностей и маски внимания

Токенизатор — часть предобученного чекпоинта. Каждый token ID соответствует строке предобученной матрицы эмбеддингов BERT. Поэтому модель нужно использовать **с тем токенизатором, на котором она обучалась**.

BERT разбивает текст на подслова WordPiece из фиксированного словаря. Редкое слово превращается в несколько фрагментов вместо неизвестного токена; префикс `##` означает продолжение предыдущего фрагмента. Мы используем модель, различающую регистр (**cased**), потому что прописные буквы помогают распознавать имена.

В батче все последовательности должны иметь одинаковую длину, поэтому короткие дополняются **padding**. Динамический padding дополняет только до длины самого длинного примера текущего батча.

`attention_mask`: 1 у настоящих токенов, 0 у заполнителей. Так внимание не учитывает padding. Это не маска для вычисления loss — в разделе 3 сравним их.

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(
    BASE_MODEL, revision=MODEL_REVISION, use_fast=True
)
batch = tokenizer(
    ["Michael studies transfer learning.", "An electroencephalographic recording."],
    padding=True,
    return_tensors="pt",
)
print({key: tuple(value.shape) for key, value in batch.items()})
for row, mask in zip(batch["input_ids"], batch["attention_mask"]):
    print(list(zip(tokenizer.convert_ids_to_tokens(row.tolist()), mask.tolist())))

**На что обратить внимание.** Частые слова остаются целыми; «electroencephalographic» разбивается на пять частей. Обе строки имеют одинаковую длину: более короткая заканчивается `[PAD]`, где маска равна 0.

Токенизатор сам добавляет `[CLS]` и `[SEP]`: в исходном тексте их нет.

BERT умеет принимать два текста одновременно, как в домашней работе с парами вопросов. Входной вектор каждого токена — сумма трёх обучаемых эмбеддингов: самого токена, сегмента (`token_type_ids`: первый или второй текст) и позиции.

BERT имеет 512 обученных позиционных эмбеддингов, поэтому более длинные входы нужно обрезать.

In [ ]:
pair = tokenizer("Can an encoder learn useful features?", "Yes, from pretraining.")
print(tokenizer.convert_ids_to_tokens(pair["input_ids"]))
print("Segments:", pair["token_type_ids"])

**На что обратить внимание.** Один `[CLS]` открывает входную последовательность, а `[SEP]` закрывает каждый текст. После первого `[SEP]` идентификатор сегмента меняется с 0 на 1. Классификатор из домашней работы использует итоговый вектор `[CLS]` для такой пары.

## 2. Предобученный энкодер и выходная голова

Предобученная модель состоит из **основы (body)** и **выходной головы (head)**.

Основа — 12-слойный энкодер BERT — выдаёт по одному контекстному вектору на токен: `[batch, sequence, hidden]`. В отличие от Word2Vec и GloVe, эти векторы зависят от **всего входа** — как слева, так и справа.

Голова превращает векторы в результаты конкретной задачи. Голова masked language model переводит каждый вектор в оценки всех слов словаря: `[batch, sequence, vocabulary]`. Для NER оставим основу и заменим голову новой, предсказывающей девять меток сущностей.

Transformers выводит отчёт о загрузке чекпоинта. Pooler и голова next-sentence prediction помечены как `UNEXPECTED`. BERT обучался двум задачам: восстановлению замаскированных токенов и предсказанию следующего предложения; чекпоинт содержит компоненты для обеих. Модель MLM использует только первую. Более поздние исследования показали, что задача next-sentence prediction не обязательна.

In [ ]:
mlm = AutoModelForMaskedLM.from_pretrained(BASE_MODEL, revision=MODEL_REVISION)
mlm.eval()
with torch.no_grad():
    representations = mlm.base_model(**batch).last_hidden_state
    vocabulary_logits = mlm(**batch).logits
print("Encoder output:", tuple(representations.shape))
print("MLM logits:", tuple(vocabulary_logits.shape))

**На что обратить внимание.** Основа выдаёт 768 чисел на токен, а MLM-голова превращает их в 28 996 оценок — по одной на элемент словаря модели cased. К той же основе можно присоединить любую голову, принимающую 768-мерные векторы.

Предобучение masked language modeling устроено так: часть входных токенов скрывается, и сеть восстанавливает их по **двустороннему контексту**. Для заполнения пропусков нужны синтаксис, семантика и некоторые сведения о мире, хотя текст обучения не размечен.

Ниже показано, как в одном шаблоне предложения меняются вероятности при замене подлежащего.

In [ ]:
fill_mask = pipeline("fill-mask", model=mlm, tokenizer=tokenizer, device=-1)
for text in ["The doctor examined the [MASK].", "The mechanic examined the [MASK]."]:
    print(text)
    print([(item["token_str"], round(item["score"], 3)) for item in fill_mask(text)])

**На что обратить внимание.** Выход — распределение вероятностей, а не один ответ: даже самый вероятный вариант может иметь лишь несколько процентов. Замена слова слева повышает вероятности тех продолжений, которые подходят контексту. Этот пример иллюстрирует принцип обучения, но несколько предсказаний ещё не доказывают, что именно «знает» модель.

In [ ]:
del fill_mask, mlm, representations, vocabulary_logits
gc.collect()

## 3. Данные NER: слова не равны токенам модели

CoNLL-2003 — предложения из новостей Reuters с разметкой четырёх типов сущностей: люди (`PER`), организации (`ORG`), места (`LOC`) и другие имена (`MISC`).

Схема IOB2: `B-X` — начало сущности типа X, `I-X` — её продолжение, `O` — вне сущности. `B-` позволяет различать две соседние сущности одного типа.

Мы берём 2048 из 14 041 обучающего предложения (около 15%) и 512 предложений для валидации. Зафиксированные ревизии, seed и подмножества делают данные и инициализацию воспроизводимыми; результаты GPU могут слегка различаться между запусками.

Эти размеры и число эпох — учебные значения, а не оценка времени выполнения. При наличии GPU используй его: BERT на CPU работает медленно. Официальная тестовая выборка не используется.

In [ ]:
raw = datasets.load_dataset("lhoestq/conll2003", revision=DATASET_REVISION)
label_names = raw["train"].features["ner_tags"].feature.names
id2label = dict(enumerate(label_names))
label2id = {label: index for index, label in id2label.items()}
train_words = raw["train"].shuffle(seed=SEED).select(range(2048))
dev_words = raw["validation"].shuffle(seed=SEED).select(range(512))
print("Labels:", label_names)
print(list(zip(train_words[0]["tokens"], train_words[0]["ner_tags"])))

**На что обратить внимание.** Метки — целые числа, индексирующие `label_names`. Большинство слов имеют метку 0 = `O`. Эти метки относятся к **словам в исходной разметке**, а не к токенам BERT.

Модель возвращает по одному вектору на подслово. По подходу из статьи BERT будем обучать модель только на **первом подслове** каждого слова. Остальные фрагменты, специальные токены и padding получают метку `-100`, игнорируемую PyTorch cross-entropy. Можно размечать все подслова, но тогда продолжения требуют меток `I-`.

Различие масок:

| Позиция | `attention_mask` | Метка |
| --- | --- | --- |
| Padding | 0 | `-100` |
| `[CLS]`, `[SEP]` | 1 | `-100` |
| Второй и следующие фрагменты слова | 1 | `-100` |
| Первый фрагмент слова | 1 | Метка исходного слова |

`attention_mask` управляет видимостью для attention; `-100` — тем, какие позиции учитывает loss. Позиция, исключённая из loss, всё равно может влиять на представления остальных токенов.

In [ ]:
def align_word_labels(word_ids, word_labels):
    aligned = []
    previous_word = None
    for word_id in word_ids:
        if word_id is None or word_id == previous_word:
            aligned.append(-100)
        else:
            aligned.append(word_labels[word_id])
        previous_word = word_id
    return aligned


def tokenize_and_align(samples):
    encoded = tokenizer(
        samples["tokens"],
        is_split_into_words=True,
        truncation=True,
        max_length=128,
    )
    encoded["labels"] = [
        align_word_labels(encoded.word_ids(batch_index=i), labels)
        for i, labels in enumerate(samples["ner_tags"])
    ]
    return encoded


assert align_word_labels([None, 0, 1, 1, 2, None], [1, 0, 5]) == [-100, 1, 0, -100, 5, -100]
example = tokenize_and_align(train_words[:1])
print(list(zip(
    tokenizer.convert_ids_to_tokens(example["input_ids"][0]),
    example["labels"][0],
)))

**На что обратить внимание.** Метка каждого слова встречается ровно один раз, у первого подслова. Продолжения вроде `##s` получают `-100`. Специальные `[CLS]` и `[SEP]` также имеют `-100`, потому что они не являются словами.

In [ ]:
train_tokens = train_words.map(
    tokenize_and_align, batched=True, remove_columns=train_words.column_names
)
dev_tokens = dev_words.map(
    tokenize_and_align, batched=True, remove_columns=dev_words.column_names
)
collator = DataCollatorForTokenClassification(tokenizer, label_pad_token_id=-100)
train_loader = DataLoader(
    train_tokens, batch_size=16, shuffle=True, collate_fn=collator,
    generator=torch.Generator().manual_seed(SEED),
)
dev_loader = DataLoader(dev_tokens, batch_size=16, shuffle=False, collate_fn=collator)
sample_batch = next(iter(train_loader))
print({key: tuple(value.shape) for key, value in sample_batch.items()})
print("Attention mask:", sample_batch["attention_mask"][0].tolist())
print("Loss labels:", sample_batch["labels"][0].tolist())

**На что обратить внимание.** В батче все тензоры одной длины — по самому длинному предложению. На padding-позициях `attention_mask` равен 0, а метка — `-100`. Внутри предложения маска везде 1, но метки пропускают специальные токены и продолжения слов.

## 4. Какие параметры обучаются?

Основа предобучена, а голова новая. `AutoModelForTokenClassification` добавляет линейный слой, который отображает каждый 768-мерный вектор токена на девять оценок меток, и случайно инициализирует его.

В отчёте загрузки классификатор помечен как `MISSING`: в чекпоинте нет его весов. Неиспользуемые головы предобученной модели помечены как `UNEXPECTED`.

Есть два основных способа адаптации: **заморозить** основу и обучать только голову либо **дообучать обе части**. Ниже мы заморозим энкодер, выполним один backward и посмотрим, куда приходят градиенты. Шага оптимизатора нет, поэтому веса не изменятся.

Три механизма, которые легко перепутать:

| Команда | Что меняет | Чего не делает |
| --- | --- | --- |
| `requires_grad_(False)` | Указывает, получает ли параметр градиенты | Не пропускает forward |
| `model.eval()` / `model.train()` | Переключает dropout и прочее поведение train/eval | Не отключает вычисление градиентов |
| `torch.no_grad()` | Запрещает autograd записывать вычисления | Не меняет поведение dropout |

Даже замороженный энкодер выполняет forward на каждом батче. Экономятся вычисления backward и состояние оптимизатора.

In [ ]:
set_seed(SEED)
model = AutoModelForTokenClassification.from_pretrained(
    BASE_MODEL,
    revision=MODEL_REVISION,
    num_labels=len(label_names),
    id2label=id2label,
    label2id=label2id,
).to(DEVICE)


def parameter_counts(model):
    return {
        "total": sum(parameter.numel() for parameter in model.parameters()),
        "trainable": sum(
            parameter.numel() for parameter in model.parameters() if parameter.requires_grad
        ),
    }


for parameter in model.base_model.parameters():
    parameter.requires_grad_(False)
model.train()
model.base_model.eval()
device_batch = {key: value.to(DEVICE) for key, value in sample_batch.items()}
model(**device_batch).loss.backward()
print("Frozen:", parameter_counts(model))
print("Encoder has gradients:", any(p.grad is not None for p in model.base_model.parameters()))
print("Head has gradients:", any(p.grad is not None for p in model.classifier.parameters()))
assert all(p.grad is None for p in model.base_model.parameters())
assert any(p.grad is not None for p in model.classifier.parameters())
model.zero_grad(set_to_none=True)
for parameter in model.base_model.parameters():
    parameter.requires_grad_(True)
print("Full fine-tuning:", parameter_counts(model))

**На что обратить внимание.** При замороженной основе обучаются только 6921 из примерно 107,7 млн параметров: матрица головы 768 × 9 и девять bias. Все знания предобучения находятся в замороженной части.

При полном fine-tuning обучаются все параметры. AdamW хранит ещё два скользящих среднего на параметр вдобавок к градиенту; обучение требует существенно больше памяти, чем сами веса.

В статье BERT на полном CoNLL-2003 фиксированные признаки почти достигли качества fine-tuning: 96,1 против 96,4 dev F1 для BERT-base. Однако там использовался обучаемый двухслойный BiLSTM поверх замороженных признаков, а модель читала целые документы. В домашнем задании сравниваем режимы на другой задаче с одной линейной головой в обоих вариантах.

## 5. Полный цикл fine-tuning

**Метрика.** Большинство слов имеют метку `O`, поэтому token accuracy остаётся высокой, даже если модель вообще не распознаёт сущности.

Используем precision, recall и F1 **на уровне сущностей** со строгим совпадением IOB2. Сущность — `B-X` с любым количеством последующих `I-X`. Предсказание считается правильным, только если совпадают тип и обе границы. `I-X`, не продолжающее сущность X, не начинает новую сущность.

Ошибка границы штрафуется дважды: как ложноположительная и как пропущенная эталонная сущность. Позиции с `-100` исключаются перед расчётом метрик — считаем слова, а не подслова.

До обучения голова инициализирована случайно, поэтому значения метрик должны быть близки к нулю.

In [ ]:
def entity_spans(tags):
    spans, start, kind = set(), None, None
    for position, tag in enumerate(tags + ["O"]):
        if start is not None and tag != f"I-{kind}":
            spans.add((kind, start, position))
            start = None
        if tag.startswith("B-"):
            start, kind = position, tag[2:]
    return spans


def entity_scores(references, predictions):
    gold_count = predicted_count = correct = 0
    for reference, prediction in zip(references, predictions):
        gold, predicted = entity_spans(reference), entity_spans(prediction)
        gold_count += len(gold)
        predicted_count += len(predicted)
        correct += len(gold & predicted)
    precision = correct / predicted_count if predicted_count else 0.0
    recall = correct / gold_count if gold_count else 0.0
    f1 = 2 * precision * recall / (precision + recall) if precision + recall else 0.0
    return {"precision": precision, "recall": recall, "f1": f1}


assert entity_spans(["B-PER", "I-PER", "O", "I-LOC", "B-ORG", "I-LOC"]) == {("PER", 0, 2), ("ORG", 4, 5)}
assert entity_scores([["B-PER", "I-PER", "B-LOC"]], [["B-PER", "O", "B-LOC"]]) == {
    "precision": 0.5, "recall": 0.5, "f1": 0.5,
}


@torch.no_grad()
def predict_word_tags(model, loader):
    model.eval()
    references, predictions = [], []
    for batch in loader:
        labels = batch["labels"]
        inputs = {key: value.to(DEVICE) for key, value in batch.items() if key != "labels"}
        predicted = model(**inputs).logits.argmax(dim=-1).cpu()
        for predicted_row, label_row in zip(predicted.tolist(), labels.tolist()):
            kept = [(prediction, label) for prediction, label in zip(predicted_row, label_row) if label != -100]
            predictions.append([id2label[prediction] for prediction, _ in kept])
            references.append([id2label[label] for _, label in kept])
    return references, predictions


def evaluate_ner(model, loader):
    return entity_scores(*predict_word_tags(model, loader))


print("Before training:", evaluate_ner(model, dev_loader))

**Обучение.** Learning rate 3e-5, размер батча 16 и 3 эпохи находятся внутри диапазонов fine-tuning, рекомендованных в статье BERT (приложение A.3).

Скорость обучения маленькая, потому что двигаются все предобученные веса: слишком большие шаги могут стереть полезные знания прежде, чем новая голова начнёт работать. Для обучения только головы обычно берут гораздо больший learning rate; в домашнем задании при заморозке используется 1e-3.

Ограничение градиента контролирует размер обновления. Проверяем validation после каждой эпохи, не подбирая гиперпараметры по test.

`train_ner` фиксирует seed и порядок данных, чтобы можно было точно повторить обучение в разделе 6.

In [ ]:
EPOCHS = 3
LEARNING_RATE = 3e-5


def train_ner(model):
    set_seed(SEED)
    train_loader.generator.manual_seed(SEED)
    optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE)
    if DEVICE.type == "cuda":
        torch.cuda.synchronize()
    started = time.perf_counter()
    for epoch in range(EPOCHS):
        model.train()
        total_loss, batches = 0.0, 0
        for batch in train_loader:
            inputs = {key: value.to(DEVICE) for key, value in batch.items()}
            optimizer.zero_grad(set_to_none=True)
            loss = model(**inputs).loss
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            total_loss += loss.item()
            batches += 1
        print({"epoch": epoch + 1, "train_loss": total_loss / batches, **evaluate_ner(model, dev_loader)})
    if DEVICE.type == "cuda":
        torch.cuda.synchronize()
    print("Training + validation seconds:", time.perf_counter() - started)


train_ner(model)

**На что обратить внимание.** Dev F1 высок уже после первой эпохи — всего 128 обновлений на 2048 предложениях. Предобученная основа предоставляет большую часть признаков, нужных NER.

Train loss падает быстрее, чем растёт dev F1: модель начинает подгоняться под обучающие предложения. На ~900 сущностях validation разница около одного пункта F1 близка к шуму.

Мы выбирали настройку, наблюдая dev-метрики, поэтому dev F1 слегка оптимистичен. Финальный результат следует получать на нетронутой тестовой выборке. Этот короткий прогон демонстрирует адаптацию, а не максимальное качество BERT для NER.

При инференсе нет меток, указывающих на первый фрагмент слова. Используй `word_ids()`, чтобы сопоставить предсказания токенов с исходными словами — так же размечается новый текст. В таблице ниже: слово, правильная метка, предсказанная метка.

In [ ]:
model.eval()
sample = dev_words[0]
encoded = tokenizer(sample["tokens"], is_split_into_words=True, return_tensors="pt", truncation=True, max_length=128)
with torch.no_grad():
    predicted = model(**{key: value.to(DEVICE) for key, value in encoded.items()}).logits.argmax(-1)[0].cpu().tolist()
previous_word = None
for token_position, word_id in enumerate(encoded.word_ids()):
    if word_id is not None and word_id != previous_word:
        print(sample["tokens"][word_id], id2label[sample["ner_tags"][word_id]], id2label[predicted[token_position]])
    previous_word = word_id

В одном предложении редко встретятся все типы ошибок, поэтому классифицируй их по всей validation-выборке.

Для каждой эталонной сущности без точного совпадения:
- Предсказанная сущность с теми же границами, но другим типом — **ошибка типа**.
- Пересекающееся предсказание с другими границами — **ошибка границы**.
- Отсутствие пересечения — **пропуск сущности**.

Предсказанные сущности, не пересекающиеся ни с одной эталонной, — false positives; эта ячейка их не перечисляет.

Обсуди по одному примеру каждой категории. После короткого обучения некоторые категории могут отсутствовать; **не придумывай примеры**.

In [ ]:
def unmatched_gold_entities(reference, prediction):
    predicted = sorted(entity_spans(prediction), key=lambda span: span[1])
    for kind, start, end in sorted(entity_spans(reference) - set(predicted), key=lambda span: span[1]):
        same_boundaries = [span for span in predicted if span[1:] == (start, end)]
        overlapping = [span for span in predicted if span[1] < end and start < span[2]]
        if same_boundaries:
            yield "wrong type", (kind, start, end), same_boundaries
        elif overlapping:
            yield "boundary", (kind, start, end), overlapping
        else:
            yield "missed", (kind, start, end), []


def span_text(words, span):
    kind, start, end = span
    return f"{kind} '{' '.join(words[start:end])}'"


dev_references, dev_predictions = predict_word_tags(model, dev_loader)
counts, examples = Counter(), {}
for index, (reference, prediction) in enumerate(zip(dev_references, dev_predictions)):
    for category, gold, predicted in unmatched_gold_entities(reference, prediction):
        counts[category] += 1
        examples.setdefault(category, (index, gold, predicted))
print("Gold entities:", sum(len(entity_spans(reference)) for reference in dev_references))
print("Without an exact match:", {category: counts[category] for category in ("missed", "wrong type", "boundary")})
for category, (index, gold, predicted) in examples.items():
    words = dev_words[index]["tokens"]
    found = ", ".join(span_text(words, span) for span in predicted) or "nothing"
    print(f"\n{category}: gold {span_text(words, gold)}, predicted {found}")
    print("   ", " ".join(words))

**На что обратить внимание.** Ошибки типа часто возникают для имён, чей тип зависит от контекста и соглашений разметки. `MISC` объединяет имена, не подходящие под другие типы.

У Reuters есть особенности стиля — например, заголовки и строки дат/мест в верхнем регистре. Эталонная разметка тоже несовершенна: в одном исследовании ошибки меток обнаружили примерно в 5% предложений тестового CoNLL-2003 ([Wang et al., 2019](https://arxiv.org/abs/1909.01441)). Иногда «ошибка» модели — это расхождение с разметкой, а не неправильное понимание текста.

## 6. Что дало предобучение?

Transfer learning предполагает, что предобучение на неразмеченном тексте помогает эффективнее использовать размеченные данные.

Чтобы увидеть эффект, создай такую же архитектуру со **случайными** весами и обучи при тех же данных, настройках и seed. Метод `from_config` создаёт модель по описанию без загрузки предобученных весов.

Это не специально оптимизированный baseline: обучение с нуля обычно потребовало бы другой настройки и гораздо больше размеченных примеров. Сравниваем только одно отличие — начальную инициализацию основы.

In [ ]:
set_seed(SEED)
scratch_config = AutoConfig.from_pretrained(
    BASE_MODEL,
    revision=MODEL_REVISION,
    num_labels=len(label_names),
    id2label=id2label,
    label2id=label2id,
)
scratch_model = AutoModelForTokenClassification.from_config(scratch_config).to(DEVICE)
print("Random initialization:", parameter_counts(scratch_model))
train_ner(scratch_model)

**На что обратить внимание.** У обеих моделей одинаковые параметры и архитектура; отличаются только начальные значения весов. Сравни dev F1 по эпохам с разделом 5. Разница покажет вклад предобучения при данных условиях: 2048 размеченных предложений и 384 обновления.

## 7. Краткая связь с T5

Наша NER-модель выдаёт ответ через новую голову с фиксированными девятью метками. T5 формирует каждый ответ **как текст**, используя выходной слой, который тоже был предобучен.

В T5 предобучение — восстановление пропусков, но **целых фрагментов**. Энкодер получает повреждённую последовательность, а декодер генерирует пропущенные фрагменты. У каждого пропуска свой sentinel-токен; цель заканчивается ещё одним sentinel. И вход энкодера, и целевая последовательность завершаются EOS.

Ниже игрушечный пример со словами: настоящий токенизатор T5 работает с подсловными ID, а не разделёнными пробелами словами. Здесь мы не загружаем и не обучаем вторую модель.

In [ ]:
source_tokens = ["The", "small", "cat", "sat", "on", "the", "warm", "mat", "."]
encoder_tokens = ["The", "<extra_id_0>", "sat", "on", "the", "<extra_id_1>", ".", "</s>"]
target_tokens = ["<extra_id_0>", "small", "cat", "<extra_id_1>", "warm", "mat", "<extra_id_2>", "</s>"]
print("Original:", " ".join(source_tokens))
print("Encoder input:", " ".join(encoder_tokens))
print("Target:", " ".join(target_tokens))

Teacher forcing: во время обучения декодер получает предыдущие **правильные** токены эталонного ответа. Его self-attention причинно-маскированный, а cross-attention может использовать представление входа. Декодер T5 начинает с pad-токена, затем получает сдвинутую вправо целевую последовательность. Loss включает sentinel-токены, пропущенные слова и EOS; метки padding игнорируются. Начальный токен — настоящая позиция декодера, даже если его ID совпадает с pad.

In [ ]:
toy_vocabulary = {"<pad>": 0, "</s>": 1}
for token in encoder_tokens + target_tokens:
    if token not in toy_vocabulary:
        toy_vocabulary[token] = len(toy_vocabulary)
labels = [toy_vocabulary[token] for token in target_tokens] + [-100, -100]
decoder_input_ids = [toy_vocabulary["<pad>"]] + [
    token_id if token_id != -100 else toy_vocabulary["<pad>"]
    for token_id in labels[:-1]
]
print("Toy labels:", labels)
print("Shifted decoder inputs:", decoder_input_ids)
print("Positions contributing to loss:", [label != -100 for label in labels])
assert len(labels) == len(decoder_input_ids)

## Проверь себя

Вопросы не оцениваются; ответы есть в разделах выше.

1. Почему предобученную модель нужно использовать с её собственным токенизатором?
2. Чем отличаются `attention_mask` и метка `-100`? Назови позицию, где маска равна 1, а метка равна `-100`.
3. Какие параметры были предобучены, какие созданы заново и сколько получает градиенты в разных режимах?
4. Что именно изменяют `requires_grad_(False)`, `model.eval()` и `torch.no_grad()`?
5. Почему F1 на уровне сущностей информативнее token accuracy для NER и почему ошибка границы штрафуется дважды?
6. Что отличалось у предобученного и случайно инициализированного запуска, а что оставалось одинаковым?

## Используй это в домашнем задании

Переиспользуй токенизацию, динамический padding, работу с устройством, проверку градиентов и цикл обучения. Вместо меток отдельных токенов теперь будет одна бинарная метка на пару вопросов.

Сравни замороженный энкодер с обучаемой головой и полный fine-tuning, начиная оба запуска с одинаковой предобученной основы и той же начальной головы.

Домашняя работа стоит 9 обязательных баллов; семинар — практика.

Источники: [руководство HF по классификации токенов](https://huggingface.co/docs/transformers/en/tasks/token_classification), [BERT](https://arxiv.org/abs/1810.04805), [T5](https://jmlr.org/papers/v21/20-074.html), [CrossWeigh](https://arxiv.org/abs/1909.01441).